# Inventario técnico de los datasets de Airbnb

## Objetivo

Examinar la estructura de los seis archivos CSV originales antes de realizar
cualquier limpieza, transformación o análisis exploratorio.

## Preguntas iniciales

- ¿Están disponibles los seis archivos esperados?
- ¿Cuántas filas y columnas tiene cada ciudad?
- ¿Comparten las mismas columnas?
- ¿Qué tipos de datos infiere Pandas?
- ¿Qué diferencias de esquema deberán resolverse posteriormente?

Los archivos originales se leerán sin modificarlos.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_directory = project_root / "data" / "raw" / "airbnb"
csv_files = sorted(data_directory.glob("*.csv"))

assert len(csv_files) == 6, (
    f"Se esperaban 6 archivos CSV, pero se encontraron {len(csv_files)}"
)

[(csv_file.name, csv_file.stat().st_size) for csv_file in csv_files]

[('london_airbnb.csv', 11578155),
 ('madrid_airbnb.csv', 2801783),
 ('milan_airbnb.csv', 2393568),
 ('NY_airbnb.csv', 7077973),
 ('sydney_airbnb.csv', 5504518),
 ('tokyo_airbnb.csv', 1738173)]

## 1. Carga controlada de los archivos

Se cargan los seis CSV completos en memoria. Pandas infiere inicialmente los
tipos de datos, pero en esta etapa no se limpia ni transforma ninguna columna.

In [2]:
datasets = {}

for csv_file in csv_files:
    datasets[csv_file.name] = pd.read_csv(
        csv_file,
        low_memory=False,
    )

len(datasets)

6

In [3]:
basic_inventory = pd.DataFrame(
    [
        {
            "file_name": file_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
        }
        for file_name, dataframe in datasets.items()
    ]
).sort_values("file_name", ignore_index=True)

basic_inventory

,file_name,rows,columns
0,NY_airbnb.csv,48895,16
1,london_airbnb.csv,85068,16
2,madrid_airbnb.csv,19618,16
3,milan_airbnb.csv,18322,15
4,sydney_airbnb.csv,36662,16
5,tokyo_airbnb.csv,11466,14


## 2. Comparación de los esquemas

Se comparan los nombres de las columnas para determinar cuáles están presentes
en todas las ciudades y cuáles aparecen solamente en algunos datasets.

En esta etapa solo se observa la estructura: no se renombran, eliminan ni crean
columnas.

In [4]:
all_columns = sorted(
    {
        column
        for dataframe in datasets.values()
        for column in dataframe.columns
    }
)

schema_matrix = pd.DataFrame(
    {
        file_name: [
            column in dataframe.columns
            for column in all_columns
        ]
        for file_name, dataframe in datasets.items()
    },
    index=all_columns,
)

schema_matrix.index.name = "column"
schema_matrix

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,True,True,True,True,True,False
calculated_host_listings_count,True,True,True,True,True,False
host_id,True,True,True,True,True,True
host_name,True,True,True,True,True,True
id,True,True,True,True,True,True
last_review,True,True,True,True,True,True
latitude,True,True,True,True,True,True
longitude,True,True,True,True,True,True
minimum_nights,True,True,True,True,True,True


In [5]:
schema_differences = schema_matrix.loc[
    ~schema_matrix.all(axis=1)
]

schema_differences

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,True,True,True,True,True,False
calculated_host_listings_count,True,True,True,True,True,False
neighbourhood_group,True,True,False,True,True,True


### Interpretación de las diferencias

Los seis datasets comparten 13 columnas. El conjunto completo contiene 16
columnas diferentes y se observaron tres diferencias de esquema:

- Tokio no contiene `availability_365`.
- Tokio no contiene `calculated_host_listings_count`.
- Milán no contiene `neighbourhood_group`.

Estas ausencias explican las diferencias en el número total de columnas. No se
consideran todavía errores de calidad, porque podrían responder a diferencias en
la información publicada para cada ciudad.

Antes de combinar los datasets será necesario decidir cómo representar estas
columnas ausentes. En este inventario no se realiza esa transformación.

In [6]:
dtype_matrix = pd.DataFrame(
    {
        file_name: dataframe.dtypes.astype(str)
        for file_name, dataframe in datasets.items()
    }
).reindex(all_columns)

dtype_matrix.index.name= "column"
dtype_matrix

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
availability_365,int64,int64,int64,int64,int64,NaN
calculated_host_listings_count,int64,int64,int64,int64,int64,NaN
host_id,int64,int64,int64,int64,int64,int64
host_name,str,str,str,str,str,str
id,int64,int64,int64,int64,int64,int64
last_review,str,str,str,str,str,str
latitude,float64,float64,float64,float64,float64,float64
longitude,float64,float64,float64,float64,float64,float64
minimum_nights,int64,int64,int64,int64,int64,int64


In [7]:
dtype_variation_count = dtype_matrix.nunique(
    axis=1,
    dropna=True,
)

dtype_differences = dtype_matrix.loc[
    dtype_variation_count > 1
]

dtype_differences

,london_airbnb.csv,madrid_airbnb.csv,milan_airbnb.csv,NY_airbnb.csv,sydney_airbnb.csv,tokyo_airbnb.csv
column,,,,,,
neighbourhood_group,float64,str,NaN,str,float64,float64


### Diferencia pendiente de validación

`neighbourhood_group` se interpreta como texto en Madrid y Nueva York, no existe
en Milán y se interpreta como `float64` en Londres, Sídney y Tokio.

La inferencia `float64` podría deberse a que esas columnas contienen únicamente
valores nulos. Esta hipótesis se comprobará mediante el porcentaje de nulos antes
de decidir si representa un problema de calidad.

## 3. Completitud de `neighbourhood_group`

Se comprueba si la diferencia de tipo inferido se explica por columnas
completamente vacías.

Para cada ciudad se registran la existencia de la columna, los valores nulos,
el porcentaje de nulos, la cantidad de valores distintos y una muestra de
valores no nulos.

In [8]:
column_to_check = "neighbourhood_group"
neighbourhood_group_profile = []

for file_name, dataframe in datasets.items():
    if column_to_check not in dataframe.columns:
        neighbourhood_group_profile.append(
            {
                "file_name": file_name,
                "column_exists": False,
                "rows": len(dataframe),
                "null_count": pd.NA,
                "null_percentage": pd.NA,
                "distinct_non_null": pd.NA,
                "sample_values": [],
            }
        )
        continue

    column = dataframe[column_to_check]
    null_count = int(column.isna().sum())

    neighbourhood_group_profile.append(
        {
            "file_name": file_name,
            "column_exists": True,
            "rows": len(dataframe),
            "null_count": null_count,
            "null_percentage": round(
                null_count / len(dataframe) * 100,
                2,
            ),
            "distinct_non_null": int(column.nunique(dropna=True)),
            "sample_values": column.dropna().unique()[:5].tolist(),
        }
    )

neighbourhood_group_profile = pd.DataFrame(
    neighbourhood_group_profile
).sort_values(
    "file_name",
    ignore_index=True,
)

neighbourhood_group_profile

,file_name,column_exists,rows,null_count,null_percentage,distinct_non_null,sample_values
0,NY_airbnb.csv,True,48895,0,0.0,5,"[Brooklyn, Manhattan, Queens, Staten Island, B..."
1,london_airbnb.csv,True,85068,85068,100.0,0,[]
2,madrid_airbnb.csv,True,19618,0,0.0,21,"[Chamartín, Latina, Arganzuela, Centro, Salama..."
3,milan_airbnb.csv,False,18322,<NA>,<NA>,<NA>,[]
4,sydney_airbnb.csv,True,36662,36662,100.0,0,[]
5,tokyo_airbnb.csv,True,11466,11466,100.0,0,[]


### Interpretación de `neighbourhood_group`

La diferencia de tipo inferido se explica por la completitud de la columna:

- Nueva York contiene 5 categorías y no tiene valores nulos.
- Madrid contiene 21 categorías y no tiene valores nulos.
- Londres, Sídney y Tokio conservan la columna, pero está completamente vacía.
- Milán no incluye la columna en su esquema.

El tipo `float64` de Londres, Sídney y Tokio es una consecuencia técnica de que
Pandas representa los valores ausentes como `NaN`; no implica que la variable
sea numérica.

Esta diferencia limita las comparaciones geográficas entre las seis ciudades.
No se imputan ni eliminan valores durante el inventario.

## 4. Perfil general de valores ausentes

Se calcula la cantidad y el porcentaje de valores nulos de cada columna presente
en cada dataset. Las columnas inexistentes ya están registradas en la comparación
de esquemas y no se confunden con columnas completamente vacías.

In [9]:
null_profile_records = []

for file_name, dataframe in datasets.items():
    for column_name in dataframe.columns:
        current_null_count = int(
            dataframe[column_name].isna().sum()
        )

        null_profile_records.append(
            {
                "file_name": file_name,
                "column_name": column_name,
                "rows": len(dataframe),
                "null_count": current_null_count,
                "null_percentage": round(
                    current_null_count / len(dataframe) * 100,
                    2,
                ),
            }
        )

null_profile = pd.DataFrame(null_profile_records)

missingness_summary = (
    null_profile.groupby("file_name")
    .agg(
        columns_present=("column_name", "count"),
        columns_with_nulls=(
            "null_count",
            lambda values: int((values > 0).sum()),
        ),
        fully_null_columns=(
            "null_percentage",
            lambda values: int((values == 100).sum()),
        ),
        maximum_null_percentage=("null_percentage", "max"),
    )
    .reset_index()
    .sort_values("file_name", ignore_index=True)
)

display(missingness_summary)

,file_name,columns_present,columns_with_nulls,fully_null_columns,maximum_null_percentage
0,NY_airbnb.csv,16,4,0,20.56
1,london_airbnb.csv,16,5,1,100.00
2,madrid_airbnb.csv,16,4,0,28.73
3,milan_airbnb.csv,15,4,0,27.63
4,sydney_airbnb.csv,16,5,1,100.00
5,tokyo_airbnb.csv,14,4,1,100.00


In [10]:
fully_null_columns = (
    null_profile.loc[
        null_profile["null_percentage"] == 100,
        [
            "file_name",
            "column_name",
            "null_count",
            "null_percentage",
        ],
    ]
    .sort_values(
        ["file_name", "column_name"],
        ignore_index=True,
    )
)

display(fully_null_columns)

,file_name,column_name,null_count,null_percentage
0,london_airbnb.csv,neighbourhood_group,85068,100.0
1,sydney_airbnb.csv,neighbourhood_group,36662,100.0
2,tokyo_airbnb.csv,neighbourhood_group,11466,100.0


In [11]:
columns_with_nulls = (
    null_profile.loc[
        null_profile["null_count"] > 0,
        [
            "file_name",
            "column_name",
            "null_count",
            "null_percentage",
        ],
    ]
    .sort_values(
        ["file_name", "null_percentage"],
        ascending=[True, False],
        ignore_index=True,
    )
)

display(columns_with_nulls)

,file_name,column_name,null_count,null_percentage
0,NY_airbnb.csv,last_review,10052,20.56
1,NY_airbnb.csv,reviews_per_month,10052,20.56
2,NY_airbnb.csv,host_name,21,0.04
3,NY_airbnb.csv,name,16,0.03
4,london_airbnb.csv,neighbourhood_group,85068,100.00
5,london_airbnb.csv,last_review,20006,23.52
6,london_airbnb.csv,reviews_per_month,20006,23.52
7,london_airbnb.csv,name,26,0.03
8,london_airbnb.csv,host_name,12,0.01
9,madrid_airbnb.csv,last_review,5637,28.73


### Patrones de valores ausentes

Se observan tres patrones:

1. `neighbourhood_group` está completamente vacía en Londres, Sídney y Tokio,
   y no existe en Milán.
2. `last_review` y `reviews_per_month` presentan exactamente la misma cantidad
   de nulos dentro de cada ciudad. Se plantea la hipótesis de que corresponden
   a alojamientos sin reseñas.
3. `name` y `host_name` tienen ausencias puntuales. Madrid presenta el mayor
   porcentaje de `host_name` ausente, con un 2,69 %.

La relación entre las columnas de reseñas se validará con `number_of_reviews`
antes de interpretar los nulos como ausencia esperada.

In [12]:
review_consistency_records = []

for file_name, dataframe in datasets.items():
    last_review_is_null = dataframe["last_review"].isna()
    reviews_per_month_is_null = dataframe["reviews_per_month"].isna()
    has_zero_reviews = dataframe["number_of_reviews"].eq(0)

    both_review_fields_null = (
        last_review_is_null
        & reviews_per_month_is_null
    )

    both_review_fields_present = (
        ~last_review_is_null
        & ~reviews_per_month_is_null
    )

    review_consistency_records.append(
        {
            "file_name": file_name,
            "zero_review_rows": int(has_zero_reviews.sum()),
            "both_review_fields_null": int(
                both_review_fields_null.sum()
            ),
            "null_pattern_disagreement": int(
                (
                    last_review_is_null
                    != reviews_per_month_is_null
                ).sum()
            ),
            "null_fields_with_positive_reviews": int(
                (
                    both_review_fields_null
                    & ~has_zero_reviews
                ).sum()
            ),
            "zero_reviews_with_fields_present": int(
                (
                    has_zero_reviews
                    & both_review_fields_present
                ).sum()
            ),
        }
    )

review_consistency = pd.DataFrame(
    review_consistency_records
).sort_values(
    "file_name",
    ignore_index=True,
)

display(review_consistency)

,file_name,zero_review_rows,both_review_fields_null,null_pattern_disagreement,null_fields_with_positive_reviews,zero_reviews_with_fields_present
0,NY_airbnb.csv,10052,10052,0,0,0
1,london_airbnb.csv,20006,20006,0,0,0
2,madrid_airbnb.csv,5637,5637,0,0,0
3,milan_airbnb.csv,5062,5062,0,0,0
4,sydney_airbnb.csv,11814,11937,0,123,0
5,tokyo_airbnb.csv,1677,1677,0,0,0


### Consistencia de la información de reseñas

En Nueva York, Londres, Madrid, Milán y Tokio, la ausencia de `last_review` y
`reviews_per_month` coincide exactamente con alojamientos que tienen cero
reseñas. En esas ciudades, los nulos son coherentes con la lógica del dataset.

Sídney presenta una excepción: 123 alojamientos tienen un
`number_of_reviews` positivo, pero no contienen `last_review` ni
`reviews_per_month`. Representan aproximadamente el 0,34 % del dataset de
Sídney y el 0,50 % de sus alojamientos con alguna reseña.

La incidencia es localizada, pero puede afectar análisis de frecuencia o
recencia de reseñas en Sídney. No se corrige durante el inventario.

In [13]:
sydney = datasets["sydney_airbnb.csv"]

sydney_review_anomalies = sydney.loc[
    sydney["number_of_reviews"].gt(0)
    & sydney["last_review"].isna()
    & sydney["reviews_per_month"].isna(),
    [
        "id",
        "name",
        "number_of_reviews",
        "last_review",
        "reviews_per_month",
    ],
].copy()

positive_review_rows = int(
    sydney["number_of_reviews"].gt(0).sum()
)

sydney_anomaly_summary = pd.DataFrame(
    {
        "metric": [
            "anomalous_rows",
            "percentage_of_all_sydney_rows",
            "percentage_of_reviewed_sydney_rows",
            "minimum_number_of_reviews",
            "median_number_of_reviews",
            "maximum_number_of_reviews",
        ],
        "value": [
            len(sydney_review_anomalies),
            round(
                len(sydney_review_anomalies)
                / len(sydney)
                * 100,
                2,
            ),
            round(
                len(sydney_review_anomalies)
                / positive_review_rows
                * 100,
                2,
            ),
            sydney_review_anomalies[
                "number_of_reviews"
            ].min(),
            sydney_review_anomalies[
                "number_of_reviews"
            ].median(),
            sydney_review_anomalies[
                "number_of_reviews"
            ].max(),
        ],
    }
)

display(sydney_anomaly_summary)
display(sydney_review_anomalies.head(10))

,metric,value
0,anomalous_rows,123.00
1,percentage_of_all_sydney_rows,0.34
2,percentage_of_reviewed_sydney_rows,0.50
3,minimum_number_of_reviews,1.00
4,median_number_of_reviews,1.00
5,maximum_number_of_reviews,4.00


,id,name,number_of_reviews,last_review,reviews_per_month
6059,8961381,South coogee home,4,NaN,NaN
8991,11512611,all amenities supplied,1,NaN,NaN
13460,16140623,Fold out bed in lounge room,1,NaN,NaN
14049,16354460,North Avalon Escape- NBH,1,NaN,NaN
15201,17257264,"PERFECT 2 BED APT - GREAT LOCATION, CLOSE TO UNSW",1,NaN,NaN
15327,17371048,Hidden gem in the heart of Surry Hills,1,NaN,NaN
17591,19629443,Comfortable and Bright private room,1,NaN,NaN
17977,19946272,Manly Waterfront Apartment,1,NaN,NaN
18484,20427620,Harbour and Opera House views,1,NaN,NaN
21047,21769946,House in Bellvue Hill,1,NaN,NaN


### Alcance de la incidencia de Sídney

Las 123 filas afectadas representan el 0,34 % del dataset completo de Sídney
y el 0,50 % de sus alojamientos con alguna reseña.

La mediana es una reseña y el máximo es cuatro, por lo que la incidencia se
concentra en alojamientos con pocas reseñas. La causa no puede determinarse con
los archivos disponibles.

El problema tiene un alcance reducido, pero debe considerarse en análisis que
utilicen `last_review` o `reviews_per_month` para Sídney.

## 5. Duplicados y posible clave de los registros

Se comprueba si existen filas completamente duplicadas y si `id` identifica de
forma única cada alojamiento dentro de cada ciudad.

También se revisa si un mismo `id` aparece en más de una ciudad, porque esto
afectaría una futura integración de los datasets.

In [14]:
duplicate_profile_records = []

for file_name, dataframe in datasets.items():
    id_is_null = dataframe["id"].isna()
    duplicated_id_mask = (
        dataframe["id"].notna()
        & dataframe["id"].duplicated(keep=False)
    )

    null_id_count = int(id_is_null.sum())
    duplicated_id_rows = int(duplicated_id_mask.sum())

    duplicate_profile_records.append(
        {
            "file_name": file_name,
            "rows": len(dataframe),
            "exact_duplicate_rows": int(
                dataframe.duplicated().sum()
            ),
            "null_id_count": null_id_count,
            "duplicated_id_rows": duplicated_id_rows,
            "duplicated_id_values": int(
                dataframe.loc[
                    duplicated_id_mask,
                    "id",
                ].nunique()
            ),
            "id_is_candidate_key": (
                null_id_count == 0
                and duplicated_id_rows == 0
            ),
        }
    )

duplicate_profile = pd.DataFrame(
    duplicate_profile_records
).sort_values(
    "file_name",
    ignore_index=True,
)

display(duplicate_profile)

,file_name,rows,exact_duplicate_rows,null_id_count,duplicated_id_rows,duplicated_id_values,id_is_candidate_key
0,NY_airbnb.csv,48895,0,0,0,0,True
1,london_airbnb.csv,85068,0,0,0,0,True
2,madrid_airbnb.csv,19618,0,0,0,0,True
3,milan_airbnb.csv,18322,0,0,0,0,True
4,sydney_airbnb.csv,36662,0,0,0,0,True
5,tokyo_airbnb.csv,11466,0,0,0,0,True


In [15]:
all_city_ids = pd.concat(
    [
        dataframe[["id"]].assign(
            file_name=file_name
        )
        for file_name, dataframe in datasets.items()
    ],
    ignore_index=True,
).dropna(subset=["id"])

city_count_per_id = (
    all_city_ids.groupby("id")["file_name"]
    .nunique()
)

cross_city_duplicate_ids = (
    city_count_per_id.loc[
        city_count_per_id > 1
    ]
    .rename("city_count")
    .reset_index()
    .sort_values(
        ["city_count", "id"],
        ascending=[False, True],
        ignore_index=True,
    )
)

cross_city_summary = pd.DataFrame(
    {
        "metric": [
            "non_null_id_rows",
            "distinct_ids",
            "ids_present_in_multiple_cities",
        ],
        "value": [
            len(all_city_ids),
            all_city_ids["id"].nunique(),
            len(cross_city_duplicate_ids),
        ],
    }
)

display(cross_city_summary)
display(cross_city_duplicate_ids.head(10))

,metric,value
0,non_null_id_rows,220031
1,distinct_ids,220031
2,ids_present_in_multiple_cities,0


,id,city_count


## 6. Conclusiones del inventario técnico

### Volumen y estructura

Los seis archivos contienen 220.031 registros:

- Londres: 85.068 filas y 16 columnas.
- Nueva York: 48.895 filas y 16 columnas.
- Sídney: 36.662 filas y 16 columnas.
- Madrid: 19.618 filas y 16 columnas.
- Milán: 18.322 filas y 15 columnas.
- Tokio: 11.466 filas y 14 columnas.

Se identificaron 16 columnas diferentes, de las cuales 13 son comunes a las
seis ciudades.

### Diferencias de esquema

- Tokio no contiene `availability_365`.
- Tokio no contiene `calculated_host_listings_count`.
- Milán no contiene `neighbourhood_group`.
- `neighbourhood_group` existe, pero está completamente vacía, en Londres,
  Sídney y Tokio.

Los datasets no deben concatenarse sin definir primero una estrategia explícita
de armonización.

### Valores ausentes

`last_review` y `reviews_per_month` faltan conjuntamente. En cinco ciudades,
esta ausencia coincide exactamente con alojamientos que no tienen reseñas.

Sídney presenta 123 alojamientos con alguna reseña, pero sin `last_review` ni
`reviews_per_month`. Representan el 0,34 % del dataset de Sídney y tienen entre
una y cuatro reseñas.

`name` y `host_name` presentan ausencias puntuales. Estas diferencias deberán
considerarse durante la especificación de limpieza.

### Duplicados y clave candidata

- No existen filas completamente duplicadas.
- `id` no tiene valores nulos.
- `id` es único dentro de cada ciudad.
- Los 220.031 registros tienen 220.031 identificadores distintos.
- No se encontraron identificadores compartidos entre ciudades.

Por tanto, `id` funciona como clave candidata para esta versión de los datos.
La ciudad deberá conservarse igualmente como atributo de procedencia durante
la futura integración.

### Evaluación

Los archivos son utilizables para continuar hacia una fase de limpieza
controlada, pero todavía no están preparados para concatenarse directamente.

Este inventario no modifica los CSV originales ni aplica reglas de limpieza,
imputación o transformación.


In [16]:
assert len(datasets) == 6
assert int(basic_inventory["rows"].sum()) == 220_031
assert len(all_columns) == 16
assert int(schema_matrix.all(axis=1).sum()) == 13

assert (
    duplicate_profile["exact_duplicate_rows"] == 0
).all()

assert (
    duplicate_profile["null_id_count"] == 0
).all()

assert duplicate_profile[
    "id_is_candidate_key"
].all()

assert len(all_city_ids) == 220_031
assert all_city_ids["id"].nunique() == 220_031
assert cross_city_duplicate_ids.empty

print("Inventario técnico validado correctamente.")

Inventario técnico validado correctamente.
